# 04 · Test runs

All methods on the **test** split (118 queries) with the settings chosen in `03_dev_tuning` (`v2/config.json`).
- **Candidate order** is shuffled per query and seed.
- **Seeds.** Order-sensitive LLM methods run with 5 shuffle seeds: 3 core seeds first, so a complete result exists early, then 2 extra seeds at the end. The other methods are order-independent; their seed only breaks ties.
- **Speed.** The LLM work is prompt-bound (1–5k tokens per call), so concurrent requests did not speed it up on this machine. They gave identical outputs in a 12-query check, but runs stay sequential.
- **Saving.** Runs are written one query at a time, so an interrupted run resumes where it stopped.

Run in the background with papermill, which saves this notebook after every cell:

```bash
cd research && .venv/bin/papermill v2/04_test_runs.ipynb v2/04_test_runs.ipynb --log-output
```

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")

RESEARCH = Path.cwd().parent if Path.cwd().name == "v2" else Path.cwd()
sys.path.insert(0, str(RESEARCH))

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

from src.figures import save, setup_style
setup_style()

In [ ]:
BENCHES = ["concept", "random"]
CORE_SEEDS = [0, 1, 2]      # complete result first: every method, both benchmarks
EXTRA_SEEDS = [3, 4]        # then more shuffle seeds for the order-sensitive LLM methods
SPLIT = "test"
POSITION_BIAS_SEEDS = [0]

In [ ]:
import json
from src.pipeline_v2 import RunConfig, run_methods

C = json.loads((RESEARCH / "v2" / "config.json").read_text())
def cfg(bench, **kw):
    base = dict(bench=bench, fields=C["fields"], max_chars=C["max_chars"], query_max_chars=C["query_max_chars"],
                split=SPLIT, stage1_k=C["stage1_k"])
    return RunConfig(**(base | kw))
C

### Lexical, embedding and cross-encoder methods

In [ ]:
for bench in BENCHES:
    display(run_methods(["bm25", "tfidf", "minilm", "specter2", "qwen", "embeddinggemma", "qwen+minilm", "bge-reranker"],
                        cfg(bench, seeds=[0])))

### Listwise LLM: full list and sliding window (3 core seeds)

In [ ]:
for bench in BENCHES:
    display(run_methods(["llama", "llama-window"], cfg(bench, seeds=CORE_SEEDS)))

### Llama + MiniLM fusion (derived from the listwise runs, no extra LLM calls)

In [ ]:
for bench in BENCHES:
    display(run_methods(["llama+minilm"], cfg(bench, seeds=CORE_SEEDS)))

### Two-stage Qwen3 → Llama (3 core seeds; stage-2 input shuffled)

In [ ]:
for bench in BENCHES:
    display(run_methods(["qwen→llama"], cfg(bench, seeds=CORE_SEEDS)))

### Pointwise LLM (order-independent, 1 seed)

In [ ]:
for bench in BENCHES:
    display(run_methods(["llama-pointwise"], cfg(bench, seeds=[0])))

### Position-bias conditions

Listwise Llama on the same test queries, with candidates in v1's positives-first order and in reversed order. Analysed in `06_analysis`.

In [ ]:
for bench in BENCHES:
    display(run_methods(["llama"], cfg(bench, orders=["original", "reversed"], seeds=POSITION_BIAS_SEEDS)))

### Extra seeds (3 and 4) for the order-sensitive LLM methods

In [ ]:
for bench in BENCHES:
    display(run_methods(["llama", "llama-window", "llama+minilm", "qwen→llama"], cfg(bench, seeds=EXTRA_SEEDS)))